# 03 — Rank This Week's Games

The actual product: refresh the current season, train the chosen model (Ridge + base features, per notebook 02) on **every** labeled game, and rank the next unplayed week.

Training on everything, including the seasons that served as test sets in notebook 02, is fine at this point. Walk-forward validation was for *choosing and estimating* the model. The final model should learn from all available history, the same way you refit on train+validation before deploying (Phase 2 — Core ML).

The logic lives in `src/rank_week.py`, so the same code runs from the command line (`venv/bin/python src/rank_week.py`) and from this notebook.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from rank_week import refresh_season, rebuild_tables, rank_week, SEASON
pd.set_option('display.max_colwidth', 80)

## 1. Refresh the current season only

Only the current season's schedule (small) is re-pulled. Play-by-play is re-pulled for this season only if a completed game is missing drama metrics. Past seasons never change, so they stay cached.

In [2]:
refresh_season(SEASON)
full = rebuild_tables()
print('Labeled games:', full.watchability_score.notna().sum())

2026 done.
Downcasting floats.


2026 done.
Downcasting floats.


Labeled games: 7047


## 2. Train on all history, score the next week

## 3. Why each game ranks where it does

For a linear model, the prediction is an exact sum: `intercept + Σ coefficient × scaled feature` (the dot product from Phase 1). So each game's score can be split into per-feature **contributions** relative to the average game. No approximation is needed, unlike tree-model explanations.

`explain()` in `src/rank_week.py` does this: multiply each scaled feature by its coefficient and show the 3 largest.

**Two kinds of explanation.** `why` shows what drives the *model* (exact Ridge contributions). `reasons` is plain language for a *reader*: division rivals, coin-flip spread, shootout total, top-8 QB by pre-game EPA/dropback, two winning teams, or a lopsided line.

QB quality is used in the reasons but **not** in the model. Tested in walk-forward, it correlates with total points (Spearman 0.22) but adds nothing to R² (0.077 → 0.077), because the Vegas spread and total already price the QBs in. It's still a real reason a person tunes in, and it's a candidate "star power" ingredient that `data/my_ratings.csv` can test once a few weeks of ratings exist.

In [3]:
board = rank_week(full)
print(f"{SEASON} Week {board.attrs['week']}")
board[['rank', 'weekday', 'away_team', 'home_team', 'spread_line', 'total_line', 'pred', 'reasons', 'why']].round(1)

2026 Week 3


,rank,weekday,away_team,home_team,spread_line,total_line,pred,reasons,why
0,1,Sunday,HOU,IND,-1.5,42.5,42.4,division rivals; Vegas calls it a coin flip (1.5-pt spread),"div_game +3.1, abs_spread +2.5, sum_season_win_pct +1.4"
1,2,Sunday,CIN,PIT,-3.5,42.5,39.7,division rivals; top QB Joe Burrow,"div_game +3.1, abs_spread +1.2, sum_season_win_pct -0.7"
2,3,Sunday,ARI,SF,8.5,47.5,38.3,division rivals; top QB Brock Purdy; but likely lopsided (8.5-pt favorite),"div_game +3.1, abs_spread -2.0, total_line +1.4"
3,4,Sunday,NE,JAX,3.0,45.5,37.7,nothing stands out,"div_game -1.8, abs_spread +1.5, sum_form_pa +1.4"
4,5,Sunday,BAL,DAL,-3.5,52.5,37.7,shootout expected (O/U 52.5); top QB Dak Prescott,"total_line +3.2, div_game -1.8, abs_spread +1.2"
5,6,Sunday,LV,NO,3.0,43.5,37.1,nothing stands out,"div_game -1.8, abs_spread +1.5, sum_season_win_pct -0.7"
6,7,Sunday,MIN,TB,-1.5,42.5,36.7,Vegas calls it a coin flip (1.5-pt spread),"abs_spread +2.5, div_game -1.8, sum_form_pa +0.8"
7,8,Sunday,CAR,CLE,-2.5,42.5,36.6,Vegas calls it a coin flip (2.5-pt spread),"abs_spread +1.8, div_game -1.8, playoff_stakes -0.6"
8,9,Sunday,LAC,BUF,7.0,50.5,36.6,shootout expected (O/U 50.5); top QB Josh Allen; but likely lopsided (7-pt f...,"total_line +2.4, div_game -1.8, gap_form_pf +1.2"
9,10,Sunday,NYJ,DET,6.5,47.5,35.9,top QB Jared Goff,"div_game -1.8, sum_form_pa -1.4, total_line +1.4"


## Reading the Week 3 board (rivalry weight 0.05)

- **Division games still lead, but narrowly:** `div_game` is now worth about +3 / −2 instead of +9 / −5.5. HOU @ IND (42.4) is only ~5 points clear of NE @ JAX and BAL @ DAL (37.7), so a strong non-division matchup can now outrank a weak division game.
- **Among the rest, Vegas sets the order:** tight spreads and high totals rise (BAL @ DAL has the week's highest total at 52.5), and lopsided games sink (KC @ MIA, −10.5).
- **Score gaps are small overall.** The model is honest that most games are coin flips for watchability, consistent with its ~0.19 within-week Spearman.
- **`primetime` gets a slightly negative coefficient.** Networks pick primetime games for audience size (big-market teams, star QBs), not for how close they'll be, so it doesn't predict drama.